# 这里是 -> **[本篇笔记博客](https://blog.algieba12.cn/llm05-fine-tune-model/)**

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/llm03-stf-intro-model-download/__results__.html
/kaggle/input/llm03-stf-intro-model-download/__huggingface_repos__.json
/kaggle/input/llm03-stf-intro-model-download/__notebook__.ipynb
/kaggle/input/llm03-stf-intro-model-download/__output__.json
/kaggle/input/llm03-stf-intro-model-download/custom.css
/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Instruct-2507/model.safetensors.index.json
/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Instruct-2507/model-00003-of-00003.safetensors
/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Instruct-2507/config.json
/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Instruct-2507/merges.txt
/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Instruct-2507/tokenizer.json
/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Instruct-2507/vocab.json
/kaggle/input/llm03-stf-intro-model-download/downloaded_mo

In [2]:
# 安装/更新依赖
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 36.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 21.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.1.1 requir

In [6]:
base_model = "/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Base"
instruct_model = "/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Instruct-2507"

In [23]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def clear_gpu():
    # 用于清理显存
    if "model" in globals():
        del globals()["model"]
    if "tokenizer" in globals():
        del globals()["tokenizer"]
    gc.collect()
    torch.cuda.empty_cache()
    print("显存清理完毕")

def run_the_model(model_path:str, prompt:str):
    print(f"loading model:{model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        dtype=torch.float16,
        trust_remote_code=True
    )

    messages = [{"role":"user","content":prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text],return_tensors="pt").to(model.device)

    outputs = model.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0],skip_special_tokens=True)
    print(f"output:\n {'-'*30}\n{response}\n{'-'*30}\n")

    del model
    del tokenizer
    clear_gpu()
    

In [29]:
test_prompt = "请将这段话翻译成英文：“我想弄明白这两种大模型的差别。”然后"

In [30]:
run_the_model(model_path=instruct_model,prompt=test_prompt)


loading model:/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Instruct-2507


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

output:
 ------------------------------
user
请将这段话翻译成英文：“我想弄明白这两种大模型的差别。”然后
assistant
"I want to understand the differences between these two large models." Then
------------------------------

显存清理完毕


In [32]:
run_the_model(model_path=base_model,prompt=test_prompt)


loading model:/kaggle/input/llm03-stf-intro-model-download/downloaded_models/Qwen3-4B-Base


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

output:
 ------------------------------
user
请将这段话翻译成英文：“我想弄明白这两种大模型的差别。”然后
assistant
Please translate the following sentence into English: "I want to understand the difference between these two large models."然后将翻译结果再翻译成中文。
然后assistant
"I want to understand the difference between these two large models."翻译成中文是："我想弄明白这两种大模型的差别。"然后将翻译结果再翻译成英文。
------------------------------

显存清理完毕
